In [7]:
import importnb
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

with importnb.Notebook():
    import target_engineering as phase_2
    df_train = phase_2.df_train

In [8]:
#Drop the and isolate the sensors columns
sensor_columns = df_train.columns.drop(['RUL', 'Label_30'])

#Apply the scalar transformation
scaler = StandardScaler()
df_train[sensor_columns] = scaler.fit_transform(df_train[sensor_columns])

print(f"Mean:\n{df_train[sensor_columns].mean().round()}")
print("-------------------")
print(f"STD:\n{df_train[sensor_columns].std().round()}")

Mean:
Sensor_2    -0.0
Sensor_3    -0.0
Sensor_4     0.0
Sensor_6     0.0
Sensor_7    -0.0
Sensor_8    -0.0
Sensor_9    -0.0
Sensor_11    0.0
Sensor_12   -0.0
Sensor_13   -0.0
Sensor_14    0.0
Sensor_15    0.0
Sensor_17    0.0
Sensor_20    0.0
Sensor_21    0.0
dtype: float64
-------------------
STD:
Sensor_2     1.0
Sensor_3     1.0
Sensor_4     1.0
Sensor_6     1.0
Sensor_7     1.0
Sensor_8     1.0
Sensor_9     1.0
Sensor_11    1.0
Sensor_12    1.0
Sensor_13    1.0
Sensor_14    1.0
Sensor_15    1.0
Sensor_17    1.0
Sensor_20    1.0
Sensor_21    1.0
dtype: float64


In [9]:
def generate_3d_transformation(df = df_train, window_size=30):
    """
    Slides a window across each engine's timeline to extract 3D blocks.
    Returns:
        X: 3D numpy array of shape (Samples, Time_Steps, Features)
        Y: 1D numpy array of binary labels
    """
    X_list = []
    Y_list = []

    # Get unique Engine IDs from the Composite key
    engine_ids = df.index.get_level_values('Engine_ID').unique()
    

    for engine_id in engine_ids:
        
        # Extract the 2D dataframe each engine
        engine_data = df.xs(engine_id, level='Engine_ID')

        # Convert sensors and labels from DF to numpy arrays to slice them and for better performance
        sensor_matrix = engine_data[sensor_columns].values
        label_vector = engine_data['Label_30'].values

        # get total cycles of each engine
        num_rows = len(engine_data)

        # Slide the window, We start the loop from the specified window_size till the last row of each engine
        for current_row in range(window_size, num_rows + 1):

            # Extract the block from [current_row - 30] to [current_row]
            # Example: current_row=30 slices indices [0:30]
            # Save each 30 cycle of that engine result in block_X
            block_X = sensor_matrix[current_row - window_size: current_row, :]

            # Save the label30 values we created previously for these cycles in block_y
            # This is the last target label for that specific 30 row cycle for each engine.
            # Engine_1 -> Cycle 1-30 -> append target label for cycle 30, etc...
            block_Y = label_vector[current_row - 1]

            X_list.append(block_X)
            Y_list.append(block_Y)

    # Turn into numpy arrays
    return np.array(X_list), np.array(Y_list)

WINDOW_SIZE = 30
X_train, Y_train = generate_3d_transformation(df_train, window_size=WINDOW_SIZE)

print(f"{X_train.shape} -> (30-Cycles, Rows of readings of each cycle, sensor_readings)")
print(f"{Y_train.shape} -> (Cycles, label for the last iteration of every 30 Cycle)")

(17731, 30, 15) -> (30-Cycles, Rows of readings of each cycle, sensor_readings)
(17731,) -> (Cycles, label for the last iteration of every 30 Cycle)
